# Daily climatology for WACCM/CAM experiments

Builds full-year daily climatologies for `U`, `T`, `Z3`, raw `O3`, and EP flux under each experiment directory. Hybrid-level `U/T/Z3/O3` are interpolated to the standard `plev` grid in Pa before averaging. Ozone-extreme composites include `low25_raw`, `low25_rm5`, `high25_raw`, and `high25_rm5`, read from the partial-O3 ranking CSV.

Default is `DEBUG_MODE=False`, which runs the full four-experiment climatology into `climatology/`. Set `DEBUG_MODE=True` for a tiny smoke-test output to `climatology_debug/`. Existing current variable files are skipped automatically; if a file already has the all-year climatology but lacks current raw/rm5 extreme composites, only those composites are updated.


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from pathlib import Path
import re
import csv
import gc
import warnings
from contextlib import nullcontext

import numpy as np
import pandas as pd
import dask
from dask.diagnostics import ProgressBar
import xarray as xr

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else nullcontext()

# ============================================================
# Daily climatology builder for WACCM/CAM long-run experiments
#
# Outputs, under each experiment directory:
#   climatology/U_climatology_plev_doy.nc
#   climatology/T_climatology_plev_doy.nc
#   climatology/O3_climatology_plev_doy.nc
#   climatology/Z3_climatology_plev_doy.nc
#   climatology/EPFLUX_climatology_plev_doy.nc
#
# Each file contains:
#   *_clim_all          : all available years climatology
#   *_clim_low25_raw    : low 25% years from raw Mar-Apr partial-O3 minimum
#   *_clim_low25_rm5    : low 25% years from 5-day-smoothed Mar-Apr partial-O3 minimum
#   *_clim_high25_raw   : high 25% years from raw Mar-Apr partial-O3 maximum
#   *_clim_high25_rm5   : high 25% years from 5-day-smoothed Mar-Apr partial-O3 maximum
#   n_days_* and n_years_* variables for later weighted averaging
#
# O3-extreme event-year alignment:
#   Ranking years are spring O3 event years. For all extreme composites, days in
#   Sep-Dec are taken from the previous model year (event_year - 1), while
#   Jan-Aug are taken from the event year. This keeps the previous winter
#   attached to the following spring ozone event.
# ============================================================

BASE_DIR = Path("/mnt/soclim0/public_data/weiji")

CASES = {
    "B2000WCN007009010011_Clim3D": {
        "root": BASE_DIR / "B2000WCN007009010011_Clim3D_timefixed",
        "ranking_case": "B2000WCN007009010011_Clim3D",
    },
    "B2000WCN_NOCOUPL001002": {
        "root": BASE_DIR / "B2000WCN_NOCOUPL001002_timefixed",
        "ranking_case": "B2000WCN_NOCOUPL",
    },
    "B2000WCN001002": {
        "root": BASE_DIR / "B2000WCN001002_timefixed",
        "ranking_case": "B2000WCN",
    },
    "BWCN": {
        "root": BASE_DIR / "BWCN",
        "ranking_case": "BWCN",
        # Split all_waves/wave1/wave2/wave_rest EP-flux files are expected.
        # Keep the legacy fallback only for old BWCN outputs if they reappear.
        "allow_legacy_epflux": True,
    },
}

# ---------------- Run control ----------------
DEBUG_MODE = False
DEBUG_CASES = ["B2000WCN001002"]
DEBUG_YEARS = [2, 3]
DEBUG_VARS = ["T"]

OVERWRITE = False
STRICT_INPUT_CHECK = True

RUN_MODEL_VARS = True
RUN_EPFLUX = True
MODEL_VARS = ["U", "T", "O3", "Z3"]
SHOW_PROGRESS = True
MODEL_VAR_SHARED_PRESSURE = True
INTERP_OUTPUT_DTYPE = np.float32

OUTPUT_SUBDIR = "climatology_debug" if DEBUG_MODE else "climatology"
CASES_TO_RUN = DEBUG_CASES if DEBUG_MODE else list(CASES)
VARS_TO_RUN = DEBUG_VARS if DEBUG_MODE else MODEL_VARS

# ---------------- Coordinates/settings ----------------
PLEV_STD_PA = np.array([
    10, 50, 100, 200, 300, 500, 1000, 2000, 3000, 5000, 7000, 10000, 15000,
    20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000
], dtype=np.float64)

DOY = np.arange(1, 366, dtype=np.int16)
MONTH_LENGTHS_NOLEAP = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], dtype=np.int16)
MONTH_ENDS_NOLEAP = np.cumsum(MONTH_LENGTHS_NOLEAP)
LOW25_PRESSURE_RANGE = "30_70hPa"
LOW25_EVENT_START_MONTH = 9
LOW25_EVENT_START_DOY = int(MONTH_ENDS_NOLEAP[LOW25_EVENT_START_MONTH - 2] + 1)

# Dask/I/O tuning for a many-core machine backed by HDD storage. Model files
# have different native chunk layouts for 3-D fields and PS, so use auto chunks
# at open time to avoid splitting stored chunks; pressure is rechunked to match
# the source variable immediately before vertical interpolation.
DASK_NUM_WORKERS = 8
DASK_CHUNK_SIZE = "96 MiB"
DASK_CONFIG = {
    "scheduler": "threads",
    "num_workers": DASK_NUM_WORKERS,
    "array.chunk-size": DASK_CHUNK_SIZE,
}
dask.config.set(DASK_CONFIG)

CHUNKS_MODEL = "auto"
CHUNKS_EP = {"time": 365, "plev": 23, "lat": 96}
NETCDF_ENGINE = "netcdf4"
COMPLEVEL = 1

YEAR_RE = re.compile(r"\.h3\.(\d{4})\.")

print("DEBUG_MODE =", DEBUG_MODE)
print("OUTPUT_SUBDIR =", OUTPUT_SUBDIR)
print("CASES_TO_RUN =", CASES_TO_RUN)
print("VARS_TO_RUN =", VARS_TO_RUN)
print("RUN_EPFLUX =", RUN_EPFLUX)
print("SHOW_PROGRESS =", SHOW_PROGRESS)
print("MODEL_VAR_SHARED_PRESSURE =", MODEL_VAR_SHARED_PRESSURE)
print("INTERP_OUTPUT_DTYPE =", INTERP_OUTPUT_DTYPE)
print("DASK_NUM_WORKERS =", DASK_NUM_WORKERS)
print("DASK_CHUNK_SIZE =", DASK_CHUNK_SIZE)
print("CHUNKS_MODEL =", CHUNKS_MODEL)
print("CHUNKS_EP =", CHUNKS_EP)


def progress_iter(iterable, **kwargs):
    if not SHOW_PROGRESS:
        return iterable
    return tqdm(iterable, **kwargs)


def dask_progress(desc=None):
    if SHOW_PROGRESS and ProgressBar is not None:
        if desc:
            print(desc)
        return ProgressBar()
    return nullcontext()


def parse_year(path):
    m = YEAR_RE.search(Path(path).name)
    if m is None:
        return None
    return int(m.group(1))


def model_calendar_from_time(time_values):
    day_index = np.asarray(time_values, dtype=float).astype(np.int64)
    year = day_index // 365 + 1
    doy = day_index % 365 + 1
    month = np.searchsorted(MONTH_ENDS_NOLEAP, doy, side="left") + 1
    return year.astype(np.int32), month.astype(np.int16), doy.astype(np.int16)


def source_year_offset_for_doy():
    # For low25 event-year composites: Sep-Dec are sampled from event_year-1.
    return np.where(DOY >= LOW25_EVENT_START_DOY, -1, 0).astype(np.int16)


def discover_var_files(case_root, var, years=None):
    files = sorted((Path(case_root) / var).glob(f"*.{var}.nc"), key=lambda p: (parse_year(p) or 10**9, p.name))
    if years is not None:
        year_set = set(int(y) for y in years)
        files = [p for p in files if parse_year(p) in year_set]
    return files


def ranking_csv_path(case_name, cfg):
    return Path(cfg["root"]) / "partial_O3" / f"{cfg['ranking_case']}_partial_O3_ranking_MarApr_min_60_90N.csv"


def load_low25_years(case_name, cfg):
    path = ranking_csv_path(case_name, cfg)
    if not path.exists():
        raise FileNotFoundError(f"Missing low25 ranking CSV for {case_name}: {path}")

    df = pd.read_csv(path)
    required = {"pressure_range", "year", "is_lowest25pct"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Ranking CSV {path} missing columns: {sorted(missing)}")

    sub = df[df["pressure_range"].astype(str).eq(LOW25_PRESSURE_RANGE)].copy()
    if sub.empty:
        raise ValueError(f"Ranking CSV {path} has no rows for pressure_range={LOW25_PRESSURE_RANGE}")

    flag = sub["is_lowest25pct"]
    if flag.dtype != bool:
        flag = flag.astype(str).str.lower().isin(["true", "1", "yes"])
    years = sorted(sub.loc[flag, "year"].astype(int).unique().tolist())
    if not years:
        raise ValueError(f"Ranking CSV {path} has no low25 years for {LOW25_PRESSURE_RANGE}")
    return years


def epflux_split_files(case_root):
    root = Path(case_root) / "EPflux_daily"
    specs = {}
    for wave in ["all_waves", "wave1", "wave2", "wave_rest"]:
        matches = sorted((root / wave).glob(f"EPFLUX_{wave}_*yr_time_plev_lat.nc"))
        if matches:
            specs[wave] = matches[-1]
    return specs


def legacy_epflux_file(case_root):
    matches = sorted((Path(case_root) / "EPflux_daily").glob("*.nc"))
    matches = [p for p in matches if p.is_file() and "EPFLUX" in p.name]
    return matches[0] if matches else None


def check_case_inputs(case_name, cfg):
    root = Path(cfg["root"])
    problems = []
    if not root.exists():
        problems.append(f"missing root: {root}")
        return problems

    for var in MODEL_VARS:
        n = len(discover_var_files(root, var))
        if n == 0:
            problems.append(f"missing {var} annual files under {root / var}")

    try:
        years = load_low25_years(case_name, cfg)
        print(f"[{case_name}] low25 years ({LOW25_PRESSURE_RANGE}): n={len(years)}, first={years[:5]}")
    except Exception as exc:
        problems.append(str(exc))

    split = epflux_split_files(root)
    if len(split) != 4:
        legacy = legacy_epflux_file(root) if cfg.get("allow_legacy_epflux", False) else None
        if legacy is None:
            problems.append(f"missing split EPflux files all_waves/wave1/wave2/wave_rest under {root / 'EPflux_daily'}")
        else:
            print(f"[{case_name}] using legacy EPflux file: {legacy}")
    else:
        print(f"[{case_name}] split EPflux files found: {', '.join(split)}")
    return problems


def validate_inputs():
    all_problems = []
    for case_name in progress_iter(CASES_TO_RUN, desc="Checking inputs", unit="case"):
        cfg = CASES[case_name]
        problems = check_case_inputs(case_name, cfg)
        if problems:
            all_problems.extend([f"{case_name}: {p}" for p in problems])
    if all_problems and STRICT_INPUT_CHECK:
        msg = "Input check failed:\n" + "\n".join(f"  - {p}" for p in all_problems)
        raise RuntimeError(msg)
    if all_problems:
        print("[WARN] Input check problems:")
        for p in all_problems:
            print("  -", p)


def rechunk_like(da, like, dims):
    if not getattr(like, "chunks", None):
        return da
    chunks = {d: like.chunksizes[d] for d in dims if d in da.dims and d in like.chunksizes}
    return da.chunk(chunks) if chunks else da


def compute_pressure_mid(ds, like=None):
    p_mid = ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]
    p_mid = p_mid.transpose("time", "lev", "lat", "lon")
    if like is not None:
        p_mid = rechunk_like(p_mid, like, ("time", "lev", "lat", "lon"))
    return p_mid


def _interp_logp_chunk(v_hyb, p_hyb, log_p_tgt, output_dtype):
    # Vectorized over all non-lev dimensions inside each dask chunk. This avoids
    # one Python np.interp call per time/lat/lon column.
    v = np.asarray(v_hyb)
    p = np.asarray(p_hyb)
    log_p = np.where((p > 0) & np.isfinite(p) & np.isfinite(v), np.log(p), np.nan)

    if log_p.shape[-1] < 2:
        return np.full(v.shape[:-1] + (len(log_p_tgt),), np.nan, dtype=output_dtype)

    # CAM hybrid mid-level pressure is ordered top-to-bottom, so pressure
    # increases with lev. Keep a defensive fallback for unexpected files.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        first = np.nanmedian(log_p[..., 0])
        last = np.nanmedian(log_p[..., -1])
    if np.isfinite(first) and np.isfinite(last) and first > last:
        log_p = log_p[..., ::-1]
        v = v[..., ::-1]

    out = np.full(v.shape[:-1] + (len(log_p_tgt),), np.nan, dtype=np.float64)
    left_p = log_p[..., :-1]
    right_p = log_p[..., 1:]
    left_v = v[..., :-1]
    right_v = v[..., 1:]

    for j, target in enumerate(log_p_tgt):
        bracket = (left_p <= target) & (target <= right_p)
        has_bracket = bracket.any(axis=-1)
        k = bracket.argmax(axis=-1)
        take_k = np.expand_dims(k, axis=-1)

        p0 = np.take_along_axis(left_p, take_k, axis=-1)[..., 0]
        p1 = np.take_along_axis(right_p, take_k, axis=-1)[..., 0]
        v0 = np.take_along_axis(left_v, take_k, axis=-1)[..., 0]
        v1 = np.take_along_axis(right_v, take_k, axis=-1)[..., 0]

        weight = (target - p0) / (p1 - p0)
        y = v0 + weight * (v1 - v0)
        out[..., j] = np.where(has_bracket & np.isfinite(y), y, np.nan)

    return out.astype(output_dtype, copy=False)


def interp_profile_logp_4d(v_hyb, p_hyb, p_tgt_pa):
    p_tgt_pa = np.asarray(p_tgt_pa, dtype=np.float64)
    out = xr.apply_ufunc(
        _interp_logp_chunk,
        v_hyb,
        p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        exclude_dims={"lev"},
        dask="parallelized",
        vectorize=False,
        output_dtypes=[INTERP_OUTPUT_DTYPE],
        kwargs={"log_p_tgt": np.log(p_tgt_pa), "output_dtype": INTERP_OUTPUT_DTYPE},
        dask_gufunc_kwargs={"output_sizes": {"plev": len(p_tgt_pa)}, "allow_rechunk": True},
    )
    return out.assign_coords(plev=("plev", p_tgt_pa))


def add_model_calendar(da):
    year, month, doy = model_calendar_from_time(da["time"].values)
    event_year = np.where(doy >= LOW25_EVENT_START_DOY, year + 1, year).astype(np.int32)
    return da.assign_coords(
        model_year=("time", year),
        model_month=("time", month),
        doy=("time", doy),
        low25_event_year=("time", event_year),
    )


def valid_time_mask(da):
    # Missing days introduced by the time-fix step are whole-day NaNs. Use a
    # representative column to count valid days without scanning every grid box.
    sample = da
    indexers = {d: int(sample.sizes[d] // 2) for d in ("lat", "lon") if d in sample.dims}
    if indexers:
        sample = sample.isel(indexers)
    reduce_dims = [d for d in sample.dims if d != "time"]
    if not reduce_dims:
        return sample.notnull()
    return sample.notnull().any(dim=reduce_dims)


def n_days_and_years_from_counts(count_by_doy):
    n_days = count_by_doy.sum("doy").astype(np.int32)
    n_years = (n_days // np.int32(365)).astype(np.int32)
    return n_days, n_years


def counts_by_doy(da):
    vt = valid_time_mask(da).astype(np.int16)
    vt = vt.assign_coords(doy=da["doy"])
    out = vt.groupby("doy").sum("time").reindex(doy=DOY, fill_value=0)
    return out.astype(np.int16)


def mean_by_doy(da):
    return da.groupby("doy").mean("time", skipna=True).reindex(doy=DOY)


def climatology_pair(da, low25_years):
    da = add_model_calendar(da)

    all_da = da
    low_mask = xr.DataArray(
        np.isin(da["low25_event_year"].values.astype(int), np.asarray(low25_years, dtype=int)),
        dims=("time",),
        coords={"time": da["time"]},
    )
    low_da = da.where(low_mask, drop=True)

    clim_all = mean_by_doy(all_da)
    clim_low = mean_by_doy(low_da)

    count_all = counts_by_doy(all_da)
    count_low = counts_by_doy(low_da)
    n_days_all, n_years_all = n_days_and_years_from_counts(count_all)
    n_days_low, n_years_low = n_days_and_years_from_counts(count_low)

    return clim_all, clim_low, n_days_all, n_years_all, n_days_low, n_years_low, count_all, count_low


def output_chunksizes_for(da):
    if da.ndim == 0:
        return None
    chunks = []
    has_lon = "lon" in da.dims
    for dim in da.dims:
        n = int(da.sizes[dim])
        if dim == "doy":
            chunks.append(min(n, 31))
        elif dim == "plev":
            chunks.append(n)
        elif dim == "lat":
            chunks.append(min(n, 48 if has_lon else 96))
        elif dim == "lon":
            chunks.append(min(n, 72))
        else:
            chunks.append(min(n, 1024))
    return tuple(chunks)


def netcdf_encoding_for(ds):
    encoding = {}
    for name, da in ds.data_vars.items():
        var_encoding = {}
        if np.issubdtype(da.dtype, np.floating):
            var_encoding.update({"zlib": True, "complevel": COMPLEVEL, "_FillValue": np.nan})
        elif da.ndim > 0:
            var_encoding.update({"zlib": True, "complevel": COMPLEVEL})
        chunksizes = output_chunksizes_for(da)
        if chunksizes is not None:
            var_encoding["chunksizes"] = chunksizes
        if var_encoding:
            encoding[name] = var_encoding
    return encoding


def prepare_dataset_atomic_write(ds, out_file, overwrite=OVERWRITE):
    out_file = Path(out_file)
    out_file.parent.mkdir(parents=True, exist_ok=True)
    if out_file.exists() and out_file.stat().st_size > 0 and not overwrite:
        print(f"[SKIP] existing: {out_file}")
        return None

    tmp = out_file.with_name(out_file.name + ".tmp")
    if tmp.exists():
        tmp.unlink()

    print(f"[WRITE] {out_file}")
    delayed = ds.to_netcdf(
        tmp,
        engine=NETCDF_ENGINE,
        format="NETCDF4",
        encoding=netcdf_encoding_for(ds),
        compute=False,
    )
    return {"dataset": ds, "tmp": tmp, "out_file": out_file, "delayed": delayed}


def finalize_atomic_write(job):
    job["tmp"].replace(job["out_file"])
    return job["out_file"]


def save_dataset_atomic(ds, out_file, overwrite=OVERWRITE):
    job = prepare_dataset_atomic_write(ds, out_file, overwrite=overwrite)
    if job is None:
        return Path(out_file)
    with dask.config.set(DASK_CONFIG), dask_progress(f"[COMPUTE] {Path(out_file).name}"):
        dask.compute(job["delayed"])
    return finalize_atomic_write(job)


def open_model_var(case_root, var, years=None):
    files = discover_var_files(case_root, var, years=years)
    if not files:
        raise FileNotFoundError(f"No {var} files found for years={years} under {case_root}")
    print(f"  opening {var}: {len(files)} files ({parse_year(files[0])}-{parse_year(files[-1])})")
    ds = xr.open_mfdataset(
        [str(p) for p in files],
        combine="nested",
        concat_dim="time",
        parallel=False,
        chunks=CHUNKS_MODEL,
        decode_times=False,
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )
    return ds, files


def model_var_output_file(case_root, var):
    return Path(case_root) / OUTPUT_SUBDIR / f"{var}_climatology_plev_doy.nc"


def choose_pressure_source_var(vars_to_run):
    for preferred in ("U", "T", "O3", "Z3"):
        if preferred in vars_to_run:
            return preferred
    return vars_to_run[0]


def validate_model_time_coords(opened):
    ref_var = next(iter(opened))
    ref_time = opened[ref_var]["time"].values
    for var, ds in opened.items():
        if not np.array_equal(ds["time"].values, ref_time):
            raise ValueError(f"time coordinate mismatch between {ref_var} and {var}; cannot share pressure safely")


def build_model_var_output(case_name, root, var, ds, files, p_mid_base, low25_years):
    source_da = ds[var].transpose("time", "lev", "lat", "lon")
    p_mid = rechunk_like(p_mid_base, source_da, ("time", "lev", "lat", "lon"))
    da = interp_profile_logp_4d(source_da, p_mid, PLEV_STD_PA).transpose("time", "plev", "lat", "lon")
    da.name = var
    da.attrs.update(source_da.attrs)
    da["plev"].attrs.update({"long_name": "pressure", "units": "Pa"})

    clim_all, clim_low, n_days_all, n_years_all, n_days_low, n_years_low, count_all, count_low = climatology_pair(da, low25_years)

    out = xr.Dataset(
        {
            f"{var}_clim_all": clim_all.astype(np.float32),
            f"{var}_clim_low25": clim_low.astype(np.float32),
            "n_days_all": n_days_all,
            "n_years_all": n_years_all,
            "n_days_low25": n_days_low,
            "n_years_low25": n_years_low,
            "n_days_all_by_doy": count_all,
            "n_days_low25_by_doy": count_low,
            "low25_event_years": xr.DataArray(np.asarray(low25_years, dtype=np.int32), dims=("low25_event_year",)),
            "source_year_offset_low25_by_doy": xr.DataArray(source_year_offset_for_doy(), dims=("doy",), coords={"doy": DOY}),
        }
    )
    out.attrs.update({
        "case_name": case_name,
        "case_root": str(root),
        "source_variable": var,
        "source_file_count": len(files),
        "source_first_year": int(parse_year(files[0])),
        "source_last_year": int(parse_year(files[-1])),
        "pressure_units": "Pa",
        "pressure_reused_across_model_vars": str(MODEL_VAR_SHARED_PRESSURE),
        "low25_definition": f"lowest 25 percent of Mar-Apr 60-90N partial O3 ranking at {LOW25_PRESSURE_RANGE}",
        "low25_event_alignment": f"doy >= {LOW25_EVENT_START_DOY} uses previous source year for each event year; doy < {LOW25_EVENT_START_DOY} uses event year",
        "debug_mode": str(DEBUG_MODE),
    })
    return out


def make_model_var_climatologies(case_name, cfg, vars_to_run, low25_years, years=None):
    root = Path(cfg["root"])
    outputs = [model_var_output_file(root, var) for var in vars_to_run]
    pending = []
    for var, out_file in zip(vars_to_run, outputs):
        if out_file.exists() and out_file.stat().st_size > 0 and not OVERWRITE:
            print(f"[{case_name}] {var}: output exists, skip")
        else:
            pending.append(var)

    if not pending:
        return outputs

    opened = {}
    files_by_var = {}
    built_datasets = []
    try:
        for var in progress_iter(pending, desc=f"{case_name} open model vars", unit="var", leave=False):
            ds, files = open_model_var(root, var, years=years)
            opened[var] = ds
            files_by_var[var] = files
        validate_model_time_coords(opened)

        pressure_var = choose_pressure_source_var(pending) if MODEL_VAR_SHARED_PRESSURE else None
        p_mid_base = None
        if MODEL_VAR_SHARED_PRESSURE:
            print(f"[{case_name}] pressure source for {pending}: {pressure_var}")
            p_mid_base = compute_pressure_mid(opened[pressure_var], like=None)

        jobs = []
        for var in progress_iter(pending, desc=f"{case_name} build model graphs", unit="var", leave=False):
            p_for_var = p_mid_base if MODEL_VAR_SHARED_PRESSURE else compute_pressure_mid(opened[var], like=None)
            out = build_model_var_output(case_name, root, var, opened[var], files_by_var[var], p_for_var, low25_years)
            built_datasets.append(out)
            job = prepare_dataset_atomic_write(out, model_var_output_file(root, var))
            if job is not None:
                jobs.append(job)

        if jobs:
            desc = f"[{case_name}] computing/writing model vars: {', '.join(pending)}"
            with dask.config.set(DASK_CONFIG), dask_progress(desc):
                dask.compute(*(job["delayed"] for job in jobs))
            for job in jobs:
                finalize_atomic_write(job)
    finally:
        for ds in built_datasets:
            ds.close()
        for ds in opened.values():
            ds.close()
        gc.collect()
    return outputs


def make_model_var_climatology(case_name, cfg, var, low25_years, years=None):
    return make_model_var_climatologies(case_name, cfg, [var], low25_years, years=years)[0]


def open_epflux_dataset(path):
    return xr.open_dataset(path, chunks=CHUNKS_EP, decode_times=False)


def epflux_var_names(ds):
    preferred = ["ep1", "ep2", "div1", "div2", "Fz"]
    return [name for name in preferred if name in ds.data_vars]


def subset_debug_years(da):
    if not DEBUG_MODE or DEBUG_YEARS is None:
        return da
    year, _, _ = model_calendar_from_time(da["time"].values)
    mask = xr.DataArray(
        np.isin(year, np.asarray(DEBUG_YEARS, dtype=int)),
        dims=("time",),
        coords={"time": da["time"]},
    )
    return da.where(mask, drop=True)


def make_epflux_climatology(case_name, cfg, low25_years):
    root = Path(cfg["root"])
    out_dir = root / OUTPUT_SUBDIR
    out_file = out_dir / "EPFLUX_climatology_plev_doy.nc"
    if out_file.exists() and out_file.stat().st_size > 0 and not OVERWRITE:
        print(f"[{case_name}] EPFLUX: output exists, skip")
        return out_file

    split = epflux_split_files(root)
    datasets = []
    source_files = []

    try:
        if len(split) == 4:
            for wave, path in progress_iter(split.items(), desc=f"{case_name} EPFLUX waves", unit="wave", leave=False):
                print(f"  opening EPFLUX {wave}: {path}")
                ds = open_epflux_dataset(path)
                source_files.append(str(path))
                for v in progress_iter(epflux_var_names(ds), desc=f"{wave} EPFLUX vars", unit="var", leave=False):
                    da = ds[v]
                    if "lat" in da.dims:
                        da = da.transpose("time", "plev", "lat")
                    else:
                        da = da.transpose("time", "plev")
                    da = subset_debug_years(da)
                    da.name = f"{v}_{wave}"
                    clim_all, clim_low, n_days_all, n_years_all, n_days_low, n_years_low, count_all, count_low = climatology_pair(da, low25_years)
                    datasets.append(xr.Dataset({
                        f"{v}_{wave}_clim_all": clim_all.astype(np.float32),
                        f"{v}_{wave}_clim_low25": clim_low.astype(np.float32),
                    }))
                ds.close()
        else:
            legacy = legacy_epflux_file(root)
            if legacy is None:
                raise FileNotFoundError(f"No EPFLUX split or legacy file found under {root / 'EPflux_daily'}")
            warnings.warn(f"{case_name}: using legacy EPFLUX file without wave split: {legacy}")
            ds = open_epflux_dataset(legacy)
            source_files.append(str(legacy))
            for v in progress_iter(epflux_var_names(ds), desc="legacy EPFLUX vars", unit="var", leave=False):
                da = ds[v]
                da = da.transpose(*[d for d in ["time", "plev", "lat"] if d in da.dims])
                da = subset_debug_years(da)
                da.name = f"{v}_all_waves"
                clim_all, clim_low, n_days_all, n_years_all, n_days_low, n_years_low, count_all, count_low = climatology_pair(da, low25_years)
                datasets.append(xr.Dataset({
                    f"{v}_all_waves_clim_all": clim_all.astype(np.float32),
                    f"{v}_all_waves_clim_low25": clim_low.astype(np.float32),
                }))
            ds.close()

        out = xr.merge(datasets, compat="override")
        out["n_days_all"] = n_days_all
        out["n_years_all"] = n_years_all
        out["n_days_low25"] = n_days_low
        out["n_years_low25"] = n_years_low
        out["n_days_all_by_doy"] = count_all
        out["n_days_low25_by_doy"] = count_low
        out["low25_event_years"] = xr.DataArray(np.asarray(low25_years, dtype=np.int32), dims=("low25_event_year",))
        out["source_year_offset_low25_by_doy"] = xr.DataArray(source_year_offset_for_doy(), dims=("doy",), coords={"doy": DOY})
        out.attrs.update({
            "case_name": case_name,
            "case_root": str(root),
            "source_files": " | ".join(source_files),
            "low25_definition": f"lowest 25 percent of Mar-Apr 60-90N partial O3 ranking at {LOW25_PRESSURE_RANGE}",
            "low25_event_alignment": f"doy >= {LOW25_EVENT_START_DOY} uses previous source year for each event year; doy < {LOW25_EVENT_START_DOY} uses event year",
            "debug_mode": str(DEBUG_MODE),
        })
        save_dataset_atomic(out, out_file)
        out.close()
    finally:
        gc.collect()
    return out_file


def process_case(case_name):
    cfg = CASES[case_name]
    root = Path(cfg["root"])
    print("\n" + "=" * 100)
    print(f"CASE: {case_name}")
    print(f"ROOT: {root}")
    print("=" * 100)

    low25_years = load_low25_years(case_name, cfg)
    years = DEBUG_YEARS if DEBUG_MODE else None

    outputs = []
    if RUN_MODEL_VARS:
        outputs.extend(make_model_var_climatologies(case_name, cfg, VARS_TO_RUN, low25_years, years=years))
    if RUN_EPFLUX:
        outputs.append(make_epflux_climatology(case_name, cfg, low25_years))
    return outputs



# -----------------------------------------------------------------------------
# Updated ozone-extreme climatology logic: low25 + high25 from 5-day-smoothed CSV
# -----------------------------------------------------------------------------
O3_EXTREME_PRESSURE_RANGE = LOW25_PRESSURE_RANGE
O3_EXTREME_SAMPLES = {
    "low25": {
        "flag_column": "is_lowest25pct",
        "description": "lowest 25 percent of 5-day-smoothed Mar-Apr 60-90N partial O3 minimum",
    },
    "high25": {
        "flag_column": "is_highest25pct",
        "description": "highest 25 percent of 5-day-smoothed Mar-Apr 60-90N partial O3 maximum",
    },
}
O3_EXTREME_SAMPLE_NAMES = ["low25", "high25"]
O3_EXTREME_RANKING_ROLLING_DAYS = 5
O3_EXTREME_RANKING_ROLLING_MIN_PERIODS = 5

# Existing all-year climatology is reused when possible. If an output file exists
# but lacks current low/high extreme composites, only the extreme composites are
# recomputed from raw daily files and merged back into the existing file.
PRESERVE_EXISTING_ALL_CLIMATOLOGY = True
OVERWRITE_EXTREME_CLIMATOLOGIES = False

print("O3_EXTREME_SAMPLE_NAMES =", O3_EXTREME_SAMPLE_NAMES)
print("O3_EXTREME_RANKING_ROLLING_DAYS =", O3_EXTREME_RANKING_ROLLING_DAYS)
print("PRESERVE_EXISTING_ALL_CLIMATOLOGY =", PRESERVE_EXISTING_ALL_CLIMATOLOGY)
print("OVERWRITE_EXTREME_CLIMATOLOGIES =", OVERWRITE_EXTREME_CLIMATOLOGIES)


def _as_bool_series(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().isin(["true", "1", "yes", "y", "t"])


def _csv_rolling_days(df):
    if "ranking_rolling_days" not in df.columns:
        return None
    vals = pd.to_numeric(df["ranking_rolling_days"], errors="coerce").dropna().unique()
    if len(vals) == 0:
        return None
    return sorted(int(v) for v in vals)


def load_o3_extreme_years(case_name, cfg, sample_name):
    if sample_name not in O3_EXTREME_SAMPLES:
        raise ValueError(f"Unsupported O3 extreme sample: {sample_name}")
    path = ranking_csv_path(case_name, cfg)
    if not path.exists():
        raise FileNotFoundError(f"Missing O3 ranking CSV for {case_name}: {path}")

    df = pd.read_csv(path)
    flag_column = O3_EXTREME_SAMPLES[sample_name]["flag_column"]
    required = {"pressure_range", "year", flag_column}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Ranking CSV {path} missing columns {sorted(missing)}. "
            "Re-run Partial_O3_with_ranking.ipynb with the 5-day low/high ranking update."
        )

    rolling_days = _csv_rolling_days(df)
    if rolling_days is None:
        print(f"[WARN] {path} has no ranking_rolling_days column; assuming legacy ranking, not 5-day smoothed.")
    elif rolling_days != [O3_EXTREME_RANKING_ROLLING_DAYS]:
        print(f"[WARN] {path} ranking_rolling_days={rolling_days}, expected {[O3_EXTREME_RANKING_ROLLING_DAYS]}")

    sub = df[df["pressure_range"].astype(str).eq(O3_EXTREME_PRESSURE_RANGE)].copy()
    if sub.empty:
        raise ValueError(f"Ranking CSV {path} has no rows for pressure_range={O3_EXTREME_PRESSURE_RANGE}")

    flag = _as_bool_series(sub[flag_column])
    years = sorted(sub.loc[flag, "year"].astype(int).unique().tolist())
    if not years:
        raise ValueError(f"Ranking CSV {path} has no {sample_name} years for {O3_EXTREME_PRESSURE_RANGE}")
    return years


def load_low25_years(case_name, cfg):
    return load_o3_extreme_years(case_name, cfg, "low25")


def load_high25_years(case_name, cfg):
    return load_o3_extreme_years(case_name, cfg, "high25")


def load_o3_extreme_years_by_sample(case_name, cfg):
    return {sample: load_o3_extreme_years(case_name, cfg, sample) for sample in O3_EXTREME_SAMPLE_NAMES}


def check_case_inputs(case_name, cfg):
    root = Path(cfg["root"])
    problems = []
    if not root.exists():
        problems.append(f"missing root: {root}")
        return problems

    for var in MODEL_VARS:
        n = len(discover_var_files(root, var))
        if n == 0:
            problems.append(f"missing {var} annual files under {root / var}")

    try:
        years_by_sample = load_o3_extreme_years_by_sample(case_name, cfg)
        for sample, years in years_by_sample.items():
            print(f"[{case_name}] {sample} years ({O3_EXTREME_PRESSURE_RANGE}): n={len(years)}, first={years[:5]}")
    except Exception as exc:
        problems.append(str(exc))

    split = epflux_split_files(root)
    if len(split) != 4:
        legacy = legacy_epflux_file(root) if cfg.get("allow_legacy_epflux", False) else None
        if legacy is None:
            problems.append(f"missing split EPflux files all_waves/wave1/wave2/wave_rest under {root / 'EPflux_daily'}")
        else:
            print(f"[{case_name}] using legacy EPflux file: {legacy}")
    else:
        print(f"[{case_name}] split EPflux files found: {', '.join(split)}")
    return problems


def add_model_calendar(da):
    year, month, doy = model_calendar_from_time(da["time"].values)
    event_year = np.where(doy >= LOW25_EVENT_START_DOY, year + 1, year).astype(np.int32)
    return da.assign_coords(
        model_year=("time", year),
        model_month=("time", month),
        doy=("time", doy),
        o3_event_year=("time", event_year),
        low25_event_year=("time", event_year),  # backward-compatible alias for older helper code
    )


def sample_mask_for_event_years(da, years):
    coord = "o3_event_year" if "o3_event_year" in da.coords else "low25_event_year"
    return xr.DataArray(
        np.isin(da[coord].values.astype(int), np.asarray(years, dtype=int)),
        dims=("time",),
        coords={"time": da["time"]},
    )


def subset_to_event_year_union(da, sample_years_by_name):
    year, month, doy = model_calendar_from_time(da["time"].values)
    event_year = np.where(doy >= LOW25_EVENT_START_DOY, year + 1, year).astype(np.int32)
    union_years = sorted({int(y) for years in sample_years_by_name.values() for y in years})
    mask = xr.DataArray(
        np.isin(event_year, np.asarray(union_years, dtype=int)),
        dims=("time",),
        coords={"time": da["time"]},
    )
    return da.where(mask, drop=True)


def climatology_all_and_extremes(da, sample_years_by_name, include_all=True, samples_to_build=None):
    da = add_model_calendar(da)
    samples_to_build = list(samples_to_build or O3_EXTREME_SAMPLE_NAMES)
    out = {}

    if include_all:
        clim_all = mean_by_doy(da)
        count_all = counts_by_doy(da)
        n_days_all, n_years_all = n_days_and_years_from_counts(count_all)
        out["all"] = {
            "clim": clim_all,
            "count": count_all,
            "n_days": n_days_all,
            "n_years": n_years_all,
        }

    for sample in samples_to_build:
        years = sample_years_by_name[sample]
        mask = sample_mask_for_event_years(da, years)
        sub = da.where(mask, drop=True)
        clim = mean_by_doy(sub)
        count = counts_by_doy(sub)
        n_days, n_years = n_days_and_years_from_counts(count)
        out[sample] = {
            "clim": clim,
            "count": count,
            "n_days": n_days,
            "n_years": n_years,
        }
    return out


def climatology_pair(da, low25_years):
    # Backward-compatible wrapper used by any old ad-hoc calls.
    results = climatology_all_and_extremes(da, {"low25": low25_years}, include_all=True, samples_to_build=["low25"])
    all_r = results["all"]
    low_r = results["low25"]
    return (
        all_r["clim"], low_r["clim"],
        all_r["n_days"], all_r["n_years"],
        low_r["n_days"], low_r["n_years"],
        all_r["count"], low_r["count"],
    )


def add_extreme_metadata(out, sample_years_by_name):
    out.attrs.update({
        "o3_extreme_pressure_range": O3_EXTREME_PRESSURE_RANGE,
        "o3_extreme_samples": ",".join(O3_EXTREME_SAMPLE_NAMES),
        "o3_extreme_ranking": "5-day running mean Mar-Apr 60-90N partial O3; low25 by annual minimum, high25 by annual maximum",
        "o3_extreme_ranking_rolling_days": int(O3_EXTREME_RANKING_ROLLING_DAYS),
        "o3_extreme_ranking_rolling_min_periods": int(O3_EXTREME_RANKING_ROLLING_MIN_PERIODS),
        "o3_extreme_event_alignment": f"doy >= {LOW25_EVENT_START_DOY} uses previous source year for each event year; doy < {LOW25_EVENT_START_DOY} uses event year",
        "low25_definition": f"{O3_EXTREME_SAMPLES['low25']['description']} at {O3_EXTREME_PRESSURE_RANGE}",
        "high25_definition": f"{O3_EXTREME_SAMPLES['high25']['description']} at {O3_EXTREME_PRESSURE_RANGE}",
    })
    return out


def add_sample_vars_to_dataset(out, prefix, results, sample_years_by_name, include_all=True, samples_to_build=None):
    samples_to_build = list(samples_to_build or O3_EXTREME_SAMPLE_NAMES)
    if include_all and "all" in results:
        out[f"{prefix}_clim_all"] = results["all"]["clim"].astype(np.float32)
        out["n_days_all"] = results["all"]["n_days"]
        out["n_years_all"] = results["all"]["n_years"]
        out["n_days_all_by_doy"] = results["all"]["count"]

    for sample in samples_to_build:
        out[f"{prefix}_clim_{sample}"] = results[sample]["clim"].astype(np.float32)
        out[f"n_days_{sample}"] = results[sample]["n_days"]
        out[f"n_years_{sample}"] = results[sample]["n_years"]
        out[f"n_days_{sample}_by_doy"] = results[sample]["count"]
        out[f"{sample}_event_years"] = xr.DataArray(
            np.asarray(sample_years_by_name[sample], dtype=np.int32),
            dims=(f"{sample}_event_year",),
        )
        out[f"source_year_offset_{sample}_by_doy"] = xr.DataArray(
            source_year_offset_for_doy(), dims=("doy",), coords={"doy": DOY}
        )
    add_extreme_metadata(out, sample_years_by_name)
    return out


def _attr_int(ds, name, default=None):
    try:
        return int(ds.attrs.get(name, default))
    except Exception:
        return default


def model_var_output_is_current(out_file, var):
    out_file = Path(out_file)
    if not out_file.exists() or out_file.stat().st_size <= 0:
        return False
    try:
        with xr.open_dataset(out_file, decode_times=False) as ds:
            if f"{var}_clim_all" not in ds.data_vars:
                return False
            if _attr_int(ds, "o3_extreme_ranking_rolling_days", None) != O3_EXTREME_RANKING_ROLLING_DAYS:
                return False
            for sample in O3_EXTREME_SAMPLE_NAMES:
                needed = [
                    f"{var}_clim_{sample}",
                    f"n_days_{sample}",
                    f"n_years_{sample}",
                    f"n_days_{sample}_by_doy",
                    f"{sample}_event_years",
                    f"source_year_offset_{sample}_by_doy",
                ]
                if any(name not in ds.variables for name in needed):
                    return False
            return True
    except Exception as exc:
        print(f"[WARN] Cannot inspect existing output {out_file}: {exc}")
        return False


def model_var_output_has_all(out_file, var):
    out_file = Path(out_file)
    if not out_file.exists() or out_file.stat().st_size <= 0:
        return False
    try:
        with xr.open_dataset(out_file, decode_times=False) as ds:
            return f"{var}_clim_all" in ds.data_vars
    except Exception:
        return False


def epflux_output_is_current(out_file):
    out_file = Path(out_file)
    if not out_file.exists() or out_file.stat().st_size <= 0:
        return False
    try:
        with xr.open_dataset(out_file, decode_times=False) as ds:
            if _attr_int(ds, "o3_extreme_ranking_rolling_days", None) != O3_EXTREME_RANKING_ROLLING_DAYS:
                return False
            all_vars = [name for name in ds.data_vars if name.endswith("_clim_all")]
            if not all_vars:
                return False
            for all_var in all_vars:
                stem = all_var[:-len("_clim_all")]
                for sample in O3_EXTREME_SAMPLE_NAMES:
                    if f"{stem}_clim_{sample}" not in ds.data_vars:
                        return False
            for sample in O3_EXTREME_SAMPLE_NAMES:
                needed = [
                    f"n_days_{sample}",
                    f"n_years_{sample}",
                    f"n_days_{sample}_by_doy",
                    f"{sample}_event_years",
                    f"source_year_offset_{sample}_by_doy",
                ]
                if any(name not in ds.variables for name in needed):
                    return False
            return True
    except Exception as exc:
        print(f"[WARN] Cannot inspect existing output {out_file}: {exc}")
        return False


def epflux_output_has_all(out_file):
    out_file = Path(out_file)
    if not out_file.exists() or out_file.stat().st_size <= 0:
        return False
    try:
        with xr.open_dataset(out_file, decode_times=False) as ds:
            return any(name.endswith("_clim_all") for name in ds.data_vars)
    except Exception:
        return False


def rewrite_existing_with_update(out_file, update_ds):
    out_file = Path(out_file)
    existing = xr.open_dataset(out_file, decode_times=False, chunks={})
    try:
        drop_names = [name for name in update_ds.data_vars if name in existing.data_vars]
        base = existing.drop_vars(drop_names) if drop_names else existing
        combined = xr.merge([base, update_ds], compat="override")
        combined.attrs.update(existing.attrs)
        combined.attrs.update(update_ds.attrs)
        job = prepare_dataset_atomic_write(combined, out_file, overwrite=True)
        with dask.config.set(DASK_CONFIG), dask_progress(f"[COMPUTE] updating extremes in {out_file.name}"):
            dask.compute(job["delayed"])
        existing.close()
        combined.close()
        return finalize_atomic_write(job)
    finally:
        try:
            existing.close()
        except Exception:
            pass


def build_model_var_output(case_name, root, var, ds, files, p_mid_base, sample_years_by_name, include_all=True, samples_to_build=None, subset_to_samples=False):
    source_da = ds[var].transpose("time", "lev", "lat", "lon")
    p_mid = rechunk_like(p_mid_base, source_da, ("time", "lev", "lat", "lon"))

    if subset_to_samples:
        source_da = subset_to_event_year_union(source_da, sample_years_by_name)
        p_mid = p_mid.sel(time=source_da["time"])

    da = interp_profile_logp_4d(source_da, p_mid, PLEV_STD_PA).transpose("time", "plev", "lat", "lon")
    da.name = var
    da.attrs.update(source_da.attrs)
    da["plev"].attrs.update({"long_name": "pressure", "units": "Pa"})

    results = climatology_all_and_extremes(da, sample_years_by_name, include_all=include_all, samples_to_build=samples_to_build)
    out = xr.Dataset()
    add_sample_vars_to_dataset(out, var, results, sample_years_by_name, include_all=include_all, samples_to_build=samples_to_build)
    out.attrs.update({
        "case_name": case_name,
        "case_root": str(root),
        "source_variable": var,
        "source_file_count": len(files),
        "source_first_year": int(parse_year(files[0])),
        "source_last_year": int(parse_year(files[-1])),
        "pressure_units": "Pa",
        "pressure_reused_across_model_vars": str(MODEL_VAR_SHARED_PRESSURE),
        "debug_mode": str(DEBUG_MODE),
    })
    return out


def make_model_var_climatologies(case_name, cfg, vars_to_run, sample_years_by_name, years=None):
    root = Path(cfg["root"])
    outputs = [model_var_output_file(root, var) for var in vars_to_run]
    full_vars = []
    update_vars = []

    for var, out_file in zip(vars_to_run, outputs):
        if model_var_output_is_current(out_file, var) and not OVERWRITE_EXTREME_CLIMATOLOGIES and not OVERWRITE:
            print(f"[{case_name}] {var}: current all/low25/high25 climatology exists, skip")
        elif PRESERVE_EXISTING_ALL_CLIMATOLOGY and model_var_output_has_all(out_file, var) and not OVERWRITE:
            print(f"[{case_name}] {var}: preserving existing all climatology; updating O3-extreme composites only")
            update_vars.append(var)
        else:
            if out_file.exists() and out_file.stat().st_size > 0 and not OVERWRITE:
                print(f"[{case_name}] {var}: output exists but cannot safely preserve all; rebuilding full file")
            full_vars.append(var)

    pending = full_vars + update_vars
    if not pending:
        return outputs

    opened = {}
    files_by_var = {}
    built_datasets = []
    try:
        for var in progress_iter(pending, desc=f"{case_name} open model vars", unit="var", leave=False):
            ds, files = open_model_var(root, var, years=years)
            opened[var] = ds
            files_by_var[var] = files
        validate_model_time_coords(opened)

        pressure_var = choose_pressure_source_var(pending) if MODEL_VAR_SHARED_PRESSURE else None
        p_mid_base = None
        if MODEL_VAR_SHARED_PRESSURE:
            print(f"[{case_name}] pressure source for {pending}: {pressure_var}")
            p_mid_base = compute_pressure_mid(opened[pressure_var], like=None)

        jobs = []
        for var in progress_iter(full_vars, desc=f"{case_name} build full model climatologies", unit="var", leave=False):
            p_for_var = p_mid_base if MODEL_VAR_SHARED_PRESSURE else compute_pressure_mid(opened[var], like=None)
            out = build_model_var_output(case_name, root, var, opened[var], files_by_var[var], p_for_var, sample_years_by_name, include_all=True, samples_to_build=O3_EXTREME_SAMPLE_NAMES, subset_to_samples=False)
            built_datasets.append(out)
            job = prepare_dataset_atomic_write(out, model_var_output_file(root, var), overwrite=True)
            if job is not None:
                jobs.append(job)

        if jobs:
            desc = f"[{case_name}] computing/writing full model vars: {', '.join(full_vars)}"
            with dask.config.set(DASK_CONFIG), dask_progress(desc):
                dask.compute(*(job["delayed"] for job in jobs))
            for job in jobs:
                finalize_atomic_write(job)

        for var in progress_iter(update_vars, desc=f"{case_name} update model extremes", unit="var", leave=False):
            p_for_var = p_mid_base if MODEL_VAR_SHARED_PRESSURE else compute_pressure_mid(opened[var], like=None)
            update_ds = build_model_var_output(case_name, root, var, opened[var], files_by_var[var], p_for_var, sample_years_by_name, include_all=False, samples_to_build=O3_EXTREME_SAMPLE_NAMES, subset_to_samples=True)
            built_datasets.append(update_ds)
            rewrite_existing_with_update(model_var_output_file(root, var), update_ds)
    finally:
        for ds in built_datasets:
            ds.close()
        for ds in opened.values():
            ds.close()
        gc.collect()
    return outputs


def make_model_var_climatology(case_name, cfg, var, sample_years_by_name, years=None):
    return make_model_var_climatologies(case_name, cfg, [var], sample_years_by_name, years=years)[0]


def make_epflux_climatology(case_name, cfg, sample_years_by_name):
    root = Path(cfg["root"])
    out_dir = root / OUTPUT_SUBDIR
    out_file = out_dir / "EPFLUX_climatology_plev_doy.nc"

    if epflux_output_is_current(out_file) and not OVERWRITE_EXTREME_CLIMATOLOGIES and not OVERWRITE:
        print(f"[{case_name}] EPFLUX: current all/low25/high25 climatology exists, skip")
        return out_file

    include_all = True
    update_existing = False
    if PRESERVE_EXISTING_ALL_CLIMATOLOGY and epflux_output_has_all(out_file) and not OVERWRITE:
        print(f"[{case_name}] EPFLUX: preserving existing all climatology; updating O3-extreme composites only")
        include_all = False
        update_existing = True

    split = epflux_split_files(root)
    datasets = []
    source_files = []
    n_days_all = n_years_all = count_all = None
    sample_counts = {}

    try:
        if len(split) == 4:
            wave_items = list(split.items())
        else:
            legacy = legacy_epflux_file(root)
            if legacy is None:
                raise FileNotFoundError(f"No EPFLUX split or legacy file found under {root / 'EPflux_daily'}")
            warnings.warn(f"{case_name}: using legacy EPFLUX file without wave split: {legacy}")
            wave_items = [("all_waves", legacy)]

        for wave, path in progress_iter(wave_items, desc=f"{case_name} EPFLUX waves", unit="wave", leave=False):
            print(f"  opening EPFLUX {wave}: {path}")
            ds = open_epflux_dataset(path)
            source_files.append(str(path))
            try:
                for v in progress_iter(epflux_var_names(ds), desc=f"{wave} EPFLUX vars", unit="var", leave=False):
                    da = ds[v]
                    if "lat" in da.dims:
                        da = da.transpose("time", "plev", "lat")
                    else:
                        da = da.transpose("time", "plev")
                    da = subset_debug_years(da)
                    if not include_all:
                        da = subset_to_event_year_union(da, sample_years_by_name)
                    da.name = f"{v}_{wave}"
                    results = climatology_all_and_extremes(da, sample_years_by_name, include_all=include_all, samples_to_build=O3_EXTREME_SAMPLE_NAMES)
                    one = xr.Dataset()
                    add_sample_vars_to_dataset(one, f"{v}_{wave}", results, sample_years_by_name, include_all=include_all, samples_to_build=O3_EXTREME_SAMPLE_NAMES)
                    datasets.append(one)

                    if include_all and n_days_all is None:
                        n_days_all = results["all"]["n_days"]
                        n_years_all = results["all"]["n_years"]
                        count_all = results["all"]["count"]
                    if not sample_counts:
                        for sample in O3_EXTREME_SAMPLE_NAMES:
                            sample_counts[sample] = {
                                "n_days": results[sample]["n_days"],
                                "n_years": results[sample]["n_years"],
                                "count": results[sample]["count"],
                            }
            finally:
                ds.close()

        out = xr.merge(datasets, compat="override")
        if include_all and n_days_all is not None:
            out["n_days_all"] = n_days_all
            out["n_years_all"] = n_years_all
            out["n_days_all_by_doy"] = count_all
        for sample in O3_EXTREME_SAMPLE_NAMES:
            out[f"n_days_{sample}"] = sample_counts[sample]["n_days"]
            out[f"n_years_{sample}"] = sample_counts[sample]["n_years"]
            out[f"n_days_{sample}_by_doy"] = sample_counts[sample]["count"]
            out[f"{sample}_event_years"] = xr.DataArray(np.asarray(sample_years_by_name[sample], dtype=np.int32), dims=(f"{sample}_event_year",))
            out[f"source_year_offset_{sample}_by_doy"] = xr.DataArray(source_year_offset_for_doy(), dims=("doy",), coords={"doy": DOY})
        out.attrs.update({
            "case_name": case_name,
            "case_root": str(root),
            "source_files": " | ".join(source_files),
            "debug_mode": str(DEBUG_MODE),
        })
        add_extreme_metadata(out, sample_years_by_name)

        if update_existing:
            rewrite_existing_with_update(out_file, out)
        else:
            save_dataset_atomic(out, out_file, overwrite=True)
        out.close()
    finally:
        gc.collect()
    return out_file


def process_case(case_name):
    cfg = CASES[case_name]
    root = Path(cfg["root"])
    print("\n" + "=" * 100)
    print(f"CASE: {case_name}")
    print(f"ROOT: {root}")
    print("=" * 100)

    sample_years_by_name = load_o3_extreme_years_by_sample(case_name, cfg)
    years = DEBUG_YEARS if DEBUG_MODE else None

    outputs = []
    if RUN_MODEL_VARS:
        outputs.extend(make_model_var_climatologies(case_name, cfg, VARS_TO_RUN, sample_years_by_name, years=years))
    if RUN_EPFLUX:
        outputs.append(make_epflux_climatology(case_name, cfg, sample_years_by_name))
    return outputs



# -----------------------------------------------------------------------------
# Final ozone-extreme climatology logic: all + raw/rm5 low25/high25 in one file.
# -----------------------------------------------------------------------------
O3_EXTREME_PRESSURE_RANGE = LOW25_PRESSURE_RANGE
O3_EXTREME_SAMPLES = {
    "low25_raw": {
        "flag_column": "is_lowest25pct_raw",
        "description": "lowest 25 percent of raw Mar-Apr 60-90N partial O3 minimum",
    },
    "low25_rm5": {
        "flag_column": "is_lowest25pct_rm5",
        "description": "lowest 25 percent of 5-day-smoothed Mar-Apr 60-90N partial O3 minimum",
    },
    "high25_raw": {
        "flag_column": "is_highest25pct_raw",
        "description": "highest 25 percent of raw Mar-Apr 60-90N partial O3 maximum",
    },
    "high25_rm5": {
        "flag_column": "is_highest25pct_rm5",
        "description": "highest 25 percent of 5-day-smoothed Mar-Apr 60-90N partial O3 maximum",
    },
}
O3_EXTREME_SAMPLE_NAMES = ["low25_raw", "low25_rm5", "high25_raw", "high25_rm5"]
O3_EXTREME_RANKING_METHODS = ["raw", "rm5"]
O3_EXTREME_RANKING_METHOD_DETAILS = {
    "raw": "unsmoothed daily partial O3",
    "rm5": "5-day running mean daily partial O3",
}

# Existing all-year climatology is reused when possible. If an output file exists
# but lacks current raw/rm5 low/high composites, only the extreme composites are
# recomputed from raw daily files and merged back into the same NetCDF file.
PRESERVE_EXISTING_ALL_CLIMATOLOGY = True
OVERWRITE_EXTREME_CLIMATOLOGIES = False
DROP_LEGACY_AMBIGUOUS_EXTREMES = True

print("O3_EXTREME_SAMPLE_NAMES =", O3_EXTREME_SAMPLE_NAMES)
print("O3_EXTREME_RANKING_METHODS =", O3_EXTREME_RANKING_METHODS)
print("PRESERVE_EXISTING_ALL_CLIMATOLOGY =", PRESERVE_EXISTING_ALL_CLIMATOLOGY)
print("OVERWRITE_EXTREME_CLIMATOLOGIES =", OVERWRITE_EXTREME_CLIMATOLOGIES)
print("DROP_LEGACY_AMBIGUOUS_EXTREMES =", DROP_LEGACY_AMBIGUOUS_EXTREMES)


def _as_bool_series(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.lower().isin(["true", "1", "yes", "y", "t"])


def load_o3_extreme_years(case_name, cfg, sample_name):
    if sample_name not in O3_EXTREME_SAMPLES:
        raise ValueError(f"Unsupported O3 extreme sample: {sample_name}")
    path = ranking_csv_path(case_name, cfg)
    if not path.exists():
        raise FileNotFoundError(f"Missing O3 ranking CSV for {case_name}: {path}")

    df = pd.read_csv(path)
    flag_column = O3_EXTREME_SAMPLES[sample_name]["flag_column"]
    required = {"pressure_range", "year", flag_column}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Ranking CSV {path} missing columns {sorted(missing)}. "
            "Re-run Partial_O3_with_ranking.ipynb so the CSV contains *_raw and *_rm5 flags."
        )

    sub = df[df["pressure_range"].astype(str).eq(O3_EXTREME_PRESSURE_RANGE)].copy()
    if sub.empty:
        raise ValueError(f"Ranking CSV {path} has no rows for pressure_range={O3_EXTREME_PRESSURE_RANGE}")

    flag = _as_bool_series(sub[flag_column])
    years = sorted(sub.loc[flag, "year"].astype(int).unique().tolist())
    if not years:
        raise ValueError(f"Ranking CSV {path} has no {sample_name} years for {O3_EXTREME_PRESSURE_RANGE}")
    return years


def load_low25_years(case_name, cfg):
    # Backward-compatible wrapper: use the Friedel-style 5-day-smoothed low25 by default.
    return load_o3_extreme_years(case_name, cfg, "low25_rm5")


def load_high25_years(case_name, cfg):
    return load_o3_extreme_years(case_name, cfg, "high25_rm5")


def load_o3_extreme_years_by_sample(case_name, cfg):
    return {sample: load_o3_extreme_years(case_name, cfg, sample) for sample in O3_EXTREME_SAMPLE_NAMES}


def check_case_inputs(case_name, cfg):
    root = Path(cfg["root"])
    problems = []
    if not root.exists():
        problems.append(f"missing root: {root}")
        return problems

    for var in MODEL_VARS:
        n = len(discover_var_files(root, var))
        if n == 0:
            problems.append(f"missing {var} annual files under {root / var}")

    try:
        years_by_sample = load_o3_extreme_years_by_sample(case_name, cfg)
        for sample, years in years_by_sample.items():
            print(f"[{case_name}] {sample} years ({O3_EXTREME_PRESSURE_RANGE}): n={len(years)}, first={years[:5]}")
    except Exception as exc:
        problems.append(str(exc))

    split = epflux_split_files(root)
    if len(split) != 4:
        legacy = legacy_epflux_file(root) if cfg.get("allow_legacy_epflux", False) else None
        if legacy is None:
            problems.append(f"missing split EPflux files all_waves/wave1/wave2/wave_rest under {root / 'EPflux_daily'}")
        else:
            print(f"[{case_name}] using legacy EPflux file: {legacy}")
    else:
        print(f"[{case_name}] split EPflux files found: {', '.join(split)}")
    return problems


def add_model_calendar(da):
    year, month, doy = model_calendar_from_time(da["time"].values)
    event_year = np.where(doy >= LOW25_EVENT_START_DOY, year + 1, year).astype(np.int32)
    return da.assign_coords(
        model_year=("time", year),
        model_month=("time", month),
        doy=("time", doy),
        o3_event_year=("time", event_year),
        low25_event_year=("time", event_year),
    )


def sample_mask_for_event_years(da, years):
    coord = "o3_event_year" if "o3_event_year" in da.coords else "low25_event_year"
    return xr.DataArray(
        np.isin(da[coord].values.astype(int), np.asarray(years, dtype=int)),
        dims=("time",),
        coords={"time": da["time"]},
    )


def subset_to_event_year_union(da, sample_years_by_name):
    year, month, doy = model_calendar_from_time(da["time"].values)
    event_year = np.where(doy >= LOW25_EVENT_START_DOY, year + 1, year).astype(np.int32)
    union_years = sorted({int(y) for years in sample_years_by_name.values() for y in years})
    mask = xr.DataArray(
        np.isin(event_year, np.asarray(union_years, dtype=int)),
        dims=("time",),
        coords={"time": da["time"]},
    )
    return da.where(mask, drop=True)


def climatology_all_and_extremes(da, sample_years_by_name, include_all=True, samples_to_build=None):
    da = add_model_calendar(da)
    samples_to_build = list(samples_to_build or O3_EXTREME_SAMPLE_NAMES)
    out = {}

    if include_all:
        clim_all = mean_by_doy(da)
        count_all = counts_by_doy(da)
        n_days_all, n_years_all = n_days_and_years_from_counts(count_all)
        out["all"] = {"clim": clim_all, "count": count_all, "n_days": n_days_all, "n_years": n_years_all}

    for sample in samples_to_build:
        years = sample_years_by_name[sample]
        sub = da.where(sample_mask_for_event_years(da, years), drop=True)
        clim = mean_by_doy(sub)
        count = counts_by_doy(sub)
        n_days, n_years = n_days_and_years_from_counts(count)
        out[sample] = {"clim": clim, "count": count, "n_days": n_days, "n_years": n_years}
    return out


def climatology_pair(da, low25_years):
    # Backward-compatible wrapper used by any old ad-hoc calls.
    results = climatology_all_and_extremes(da, {"low25_rm5": low25_years}, include_all=True, samples_to_build=["low25_rm5"])
    all_r = results["all"]
    low_r = results["low25_rm5"]
    return (
        all_r["clim"], low_r["clim"],
        all_r["n_days"], all_r["n_years"],
        low_r["n_days"], low_r["n_years"],
        all_r["count"], low_r["count"],
    )


def add_extreme_metadata(out, sample_years_by_name):
    out.attrs.update({
        "o3_extreme_pressure_range": O3_EXTREME_PRESSURE_RANGE,
        "o3_extreme_samples": ",".join(O3_EXTREME_SAMPLE_NAMES),
        "o3_extreme_ranking_methods": ",".join(O3_EXTREME_RANKING_METHODS),
        "o3_extreme_ranking": "raw and 5-day-running-mean Mar-Apr 60-90N partial O3; low25 by annual minimum, high25 by annual maximum",
        "o3_extreme_event_alignment": f"doy >= {LOW25_EVENT_START_DOY} uses previous source year for each event year; doy < {LOW25_EVENT_START_DOY} uses event year",
    })
    for sample, spec in O3_EXTREME_SAMPLES.items():
        out.attrs[f"{sample}_definition"] = f"{spec['description']} at {O3_EXTREME_PRESSURE_RANGE}"
    return out


def add_sample_vars_to_dataset(out, prefix, results, sample_years_by_name, include_all=True, samples_to_build=None):
    samples_to_build = list(samples_to_build or O3_EXTREME_SAMPLE_NAMES)
    if include_all and "all" in results:
        out[f"{prefix}_clim_all"] = results["all"]["clim"].astype(np.float32)
        out["n_days_all"] = results["all"]["n_days"]
        out["n_years_all"] = results["all"]["n_years"]
        out["n_days_all_by_doy"] = results["all"]["count"]

    for sample in samples_to_build:
        out[f"{prefix}_clim_{sample}"] = results[sample]["clim"].astype(np.float32)
        out[f"n_days_{sample}"] = results[sample]["n_days"]
        out[f"n_years_{sample}"] = results[sample]["n_years"]
        out[f"n_days_{sample}_by_doy"] = results[sample]["count"]
        out[f"{sample}_event_years"] = xr.DataArray(np.asarray(sample_years_by_name[sample], dtype=np.int32), dims=(f"{sample}_event_year",))
        out[f"source_year_offset_{sample}_by_doy"] = xr.DataArray(source_year_offset_for_doy(), dims=("doy",), coords={"doy": DOY})
    add_extreme_metadata(out, sample_years_by_name)
    return out


def _attr_csv_set(ds, name):
    return {s.strip() for s in str(ds.attrs.get(name, "")).split(",") if s.strip()}


def model_var_output_is_current(out_file, var):
    out_file = Path(out_file)
    if not out_file.exists() or out_file.stat().st_size <= 0:
        return False
    try:
        with xr.open_dataset(out_file, decode_times=False) as ds:
            if f"{var}_clim_all" not in ds.data_vars:
                return False
            if _attr_csv_set(ds, "o3_extreme_samples") != set(O3_EXTREME_SAMPLE_NAMES):
                return False
            if _attr_csv_set(ds, "o3_extreme_ranking_methods") != set(O3_EXTREME_RANKING_METHODS):
                return False
            for sample in O3_EXTREME_SAMPLE_NAMES:
                needed = [
                    f"{var}_clim_{sample}",
                    f"n_days_{sample}",
                    f"n_years_{sample}",
                    f"n_days_{sample}_by_doy",
                    f"{sample}_event_years",
                    f"source_year_offset_{sample}_by_doy",
                ]
                if any(name not in ds.variables for name in needed):
                    return False
            return True
    except Exception as exc:
        print(f"[WARN] Cannot inspect existing output {out_file}: {exc}")
        return False


def model_var_output_has_all(out_file, var):
    out_file = Path(out_file)
    if not out_file.exists() or out_file.stat().st_size <= 0:
        return False
    try:
        with xr.open_dataset(out_file, decode_times=False) as ds:
            return f"{var}_clim_all" in ds.data_vars
    except Exception:
        return False


def epflux_output_is_current(out_file):
    out_file = Path(out_file)
    if not out_file.exists() or out_file.stat().st_size <= 0:
        return False
    try:
        with xr.open_dataset(out_file, decode_times=False) as ds:
            if _attr_csv_set(ds, "o3_extreme_samples") != set(O3_EXTREME_SAMPLE_NAMES):
                return False
            if _attr_csv_set(ds, "o3_extreme_ranking_methods") != set(O3_EXTREME_RANKING_METHODS):
                return False
            all_vars = [name for name in ds.data_vars if name.endswith("_clim_all")]
            if not all_vars:
                return False
            for all_var in all_vars:
                stem = all_var[:-len("_clim_all")]
                for sample in O3_EXTREME_SAMPLE_NAMES:
                    if f"{stem}_clim_{sample}" not in ds.data_vars:
                        return False
            for sample in O3_EXTREME_SAMPLE_NAMES:
                needed = [
                    f"n_days_{sample}",
                    f"n_years_{sample}",
                    f"n_days_{sample}_by_doy",
                    f"{sample}_event_years",
                    f"source_year_offset_{sample}_by_doy",
                ]
                if any(name not in ds.variables for name in needed):
                    return False
            return True
    except Exception as exc:
        print(f"[WARN] Cannot inspect existing output {out_file}: {exc}")
        return False


def epflux_output_has_all(out_file):
    out_file = Path(out_file)
    if not out_file.exists() or out_file.stat().st_size <= 0:
        return False
    try:
        with xr.open_dataset(out_file, decode_times=False) as ds:
            return any(name.endswith("_clim_all") for name in ds.data_vars)
    except Exception:
        return False


def legacy_ambiguous_extreme_names(ds):
    if not DROP_LEGACY_AMBIGUOUS_EXTREMES:
        return []
    drop = []
    legacy_exact = {
        "n_days_low25", "n_years_low25", "n_days_low25_by_doy", "low25_event_years", "source_year_offset_low25_by_doy",
        "n_days_high25", "n_years_high25", "n_days_high25_by_doy", "high25_event_years", "source_year_offset_high25_by_doy",
    }
    for name in ds.variables:
        if name in legacy_exact:
            drop.append(name)
        if name.endswith("_clim_low25") or name.endswith("_clim_high25"):
            drop.append(name)
    return sorted(set(drop))


def rewrite_existing_with_update(out_file, update_ds):
    out_file = Path(out_file)
    existing = xr.open_dataset(out_file, decode_times=False, chunks={})
    try:
        drop_names = [name for name in update_ds.data_vars if name in existing.data_vars]
        drop_names.extend(legacy_ambiguous_extreme_names(existing))
        drop_names = sorted(set(drop_names))
        base = existing.drop_vars(drop_names) if drop_names else existing
        combined = xr.merge([base, update_ds], compat="override")
        combined.attrs.update(existing.attrs)
        combined.attrs.update(update_ds.attrs)
        job = prepare_dataset_atomic_write(combined, out_file, overwrite=True)
        with dask.config.set(DASK_CONFIG), dask_progress(f"[COMPUTE] updating raw/rm5 extremes in {out_file.name}"):
            dask.compute(job["delayed"])
        existing.close()
        combined.close()
        return finalize_atomic_write(job)
    finally:
        try:
            existing.close()
        except Exception:
            pass


def build_model_var_output(case_name, root, var, ds, files, p_mid_base, sample_years_by_name, include_all=True, samples_to_build=None, subset_to_samples=False):
    source_da = ds[var].transpose("time", "lev", "lat", "lon")
    p_mid = rechunk_like(p_mid_base, source_da, ("time", "lev", "lat", "lon"))

    if subset_to_samples:
        source_da = subset_to_event_year_union(source_da, sample_years_by_name)
        p_mid = p_mid.sel(time=source_da["time"])

    da = interp_profile_logp_4d(source_da, p_mid, PLEV_STD_PA).transpose("time", "plev", "lat", "lon")
    da.name = var
    da.attrs.update(source_da.attrs)
    da["plev"].attrs.update({"long_name": "pressure", "units": "Pa"})

    results = climatology_all_and_extremes(da, sample_years_by_name, include_all=include_all, samples_to_build=samples_to_build)
    out = xr.Dataset()
    add_sample_vars_to_dataset(out, var, results, sample_years_by_name, include_all=include_all, samples_to_build=samples_to_build)
    out.attrs.update({
        "case_name": case_name,
        "case_root": str(root),
        "source_variable": var,
        "source_file_count": len(files),
        "source_first_year": int(parse_year(files[0])),
        "source_last_year": int(parse_year(files[-1])),
        "pressure_units": "Pa",
        "pressure_reused_across_model_vars": str(MODEL_VAR_SHARED_PRESSURE),
        "debug_mode": str(DEBUG_MODE),
    })
    return out


def make_model_var_climatologies(case_name, cfg, vars_to_run, sample_years_by_name, years=None):
    root = Path(cfg["root"])
    outputs = [model_var_output_file(root, var) for var in vars_to_run]
    full_vars = []
    update_vars = []

    for var, out_file in zip(vars_to_run, outputs):
        if model_var_output_is_current(out_file, var) and not OVERWRITE_EXTREME_CLIMATOLOGIES and not OVERWRITE:
            print(f"[{case_name}] {var}: current all/raw/rm5 low/high climatology exists, skip")
        elif PRESERVE_EXISTING_ALL_CLIMATOLOGY and model_var_output_has_all(out_file, var) and not OVERWRITE:
            print(f"[{case_name}] {var}: preserving existing all climatology; updating raw/rm5 O3-extreme composites only")
            update_vars.append(var)
        else:
            if out_file.exists() and out_file.stat().st_size > 0 and not OVERWRITE:
                print(f"[{case_name}] {var}: output exists but cannot safely preserve all; rebuilding full file")
            full_vars.append(var)

    pending = full_vars + update_vars
    if not pending:
        return outputs

    opened = {}
    files_by_var = {}
    built_datasets = []
    try:
        for var in progress_iter(pending, desc=f"{case_name} open model vars", unit="var", leave=False):
            ds, files = open_model_var(root, var, years=years)
            opened[var] = ds
            files_by_var[var] = files
        validate_model_time_coords(opened)

        pressure_var = choose_pressure_source_var(pending) if MODEL_VAR_SHARED_PRESSURE else None
        p_mid_base = None
        if MODEL_VAR_SHARED_PRESSURE:
            print(f"[{case_name}] pressure source for {pending}: {pressure_var}")
            p_mid_base = compute_pressure_mid(opened[pressure_var], like=None)

        jobs = []
        for var in progress_iter(full_vars, desc=f"{case_name} build full model climatologies", unit="var", leave=False):
            p_for_var = p_mid_base if MODEL_VAR_SHARED_PRESSURE else compute_pressure_mid(opened[var], like=None)
            out = build_model_var_output(case_name, root, var, opened[var], files_by_var[var], p_for_var, sample_years_by_name, include_all=True, samples_to_build=O3_EXTREME_SAMPLE_NAMES, subset_to_samples=False)
            built_datasets.append(out)
            job = prepare_dataset_atomic_write(out, model_var_output_file(root, var), overwrite=True)
            if job is not None:
                jobs.append(job)

        if jobs:
            desc = f"[{case_name}] computing/writing full model vars: {', '.join(full_vars)}"
            with dask.config.set(DASK_CONFIG), dask_progress(desc):
                dask.compute(*(job["delayed"] for job in jobs))
            for job in jobs:
                finalize_atomic_write(job)

        for var in progress_iter(update_vars, desc=f"{case_name} update model raw/rm5 extremes", unit="var", leave=False):
            p_for_var = p_mid_base if MODEL_VAR_SHARED_PRESSURE else compute_pressure_mid(opened[var], like=None)
            update_ds = build_model_var_output(case_name, root, var, opened[var], files_by_var[var], p_for_var, sample_years_by_name, include_all=False, samples_to_build=O3_EXTREME_SAMPLE_NAMES, subset_to_samples=True)
            built_datasets.append(update_ds)
            rewrite_existing_with_update(model_var_output_file(root, var), update_ds)
    finally:
        for ds in built_datasets:
            ds.close()
        for ds in opened.values():
            ds.close()
        gc.collect()
    return outputs


def make_model_var_climatology(case_name, cfg, var, sample_years_by_name, years=None):
    return make_model_var_climatologies(case_name, cfg, [var], sample_years_by_name, years=years)[0]


def make_epflux_climatology(case_name, cfg, sample_years_by_name):
    root = Path(cfg["root"])
    out_dir = root / OUTPUT_SUBDIR
    out_file = out_dir / "EPFLUX_climatology_plev_doy.nc"

    if epflux_output_is_current(out_file) and not OVERWRITE_EXTREME_CLIMATOLOGIES and not OVERWRITE:
        print(f"[{case_name}] EPFLUX: current all/raw/rm5 low/high climatology exists, skip")
        return out_file

    include_all = True
    update_existing = False
    if PRESERVE_EXISTING_ALL_CLIMATOLOGY and epflux_output_has_all(out_file) and not OVERWRITE:
        print(f"[{case_name}] EPFLUX: preserving existing all climatology; updating raw/rm5 O3-extreme composites only")
        include_all = False
        update_existing = True

    split = epflux_split_files(root)
    datasets = []
    source_files = []
    n_days_all = n_years_all = count_all = None
    sample_counts = {}

    try:
        if len(split) == 4:
            wave_items = list(split.items())
        else:
            legacy = legacy_epflux_file(root)
            if legacy is None:
                raise FileNotFoundError(f"No EPFLUX split or legacy file found under {root / 'EPflux_daily'}")
            warnings.warn(f"{case_name}: using legacy EPFLUX file without wave split: {legacy}")
            wave_items = [("all_waves", legacy)]

        for wave, path in progress_iter(wave_items, desc=f"{case_name} EPFLUX waves", unit="wave", leave=False):
            print(f"  opening EPFLUX {wave}: {path}")
            ds = open_epflux_dataset(path)
            source_files.append(str(path))
            try:
                for v in progress_iter(epflux_var_names(ds), desc=f"{wave} EPFLUX vars", unit="var", leave=False):
                    da = ds[v]
                    if "lat" in da.dims:
                        da = da.transpose("time", "plev", "lat")
                    else:
                        da = da.transpose("time", "plev")
                    da = subset_debug_years(da)
                    if not include_all:
                        da = subset_to_event_year_union(da, sample_years_by_name)
                    da.name = f"{v}_{wave}"
                    results = climatology_all_and_extremes(da, sample_years_by_name, include_all=include_all, samples_to_build=O3_EXTREME_SAMPLE_NAMES)
                    one = xr.Dataset()
                    add_sample_vars_to_dataset(one, f"{v}_{wave}", results, sample_years_by_name, include_all=include_all, samples_to_build=O3_EXTREME_SAMPLE_NAMES)
                    datasets.append(one)

                    if include_all and n_days_all is None:
                        n_days_all = results["all"]["n_days"]
                        n_years_all = results["all"]["n_years"]
                        count_all = results["all"]["count"]
                    if not sample_counts:
                        for sample in O3_EXTREME_SAMPLE_NAMES:
                            sample_counts[sample] = {
                                "n_days": results[sample]["n_days"],
                                "n_years": results[sample]["n_years"],
                                "count": results[sample]["count"],
                            }
            finally:
                ds.close()

        out = xr.merge(datasets, compat="override")
        if include_all and n_days_all is not None:
            out["n_days_all"] = n_days_all
            out["n_years_all"] = n_years_all
            out["n_days_all_by_doy"] = count_all
        for sample in O3_EXTREME_SAMPLE_NAMES:
            out[f"n_days_{sample}"] = sample_counts[sample]["n_days"]
            out[f"n_years_{sample}"] = sample_counts[sample]["n_years"]
            out[f"n_days_{sample}_by_doy"] = sample_counts[sample]["count"]
            out[f"{sample}_event_years"] = xr.DataArray(np.asarray(sample_years_by_name[sample], dtype=np.int32), dims=(f"{sample}_event_year",))
            out[f"source_year_offset_{sample}_by_doy"] = xr.DataArray(source_year_offset_for_doy(), dims=("doy",), coords={"doy": DOY})
        out.attrs.update({"case_name": case_name, "case_root": str(root), "source_files": " | ".join(source_files), "debug_mode": str(DEBUG_MODE)})
        add_extreme_metadata(out, sample_years_by_name)

        if update_existing:
            rewrite_existing_with_update(out_file, out)
        else:
            save_dataset_atomic(out, out_file, overwrite=True)
        out.close()
    finally:
        gc.collect()
    return out_file


def process_case(case_name):
    cfg = CASES[case_name]
    root = Path(cfg["root"])
    print("\n" + "=" * 100)
    print(f"CASE: {case_name}")
    print(f"ROOT: {root}")
    print("=" * 100)

    sample_years_by_name = load_o3_extreme_years_by_sample(case_name, cfg)
    years = DEBUG_YEARS if DEBUG_MODE else None

    outputs = []
    if RUN_MODEL_VARS:
        outputs.extend(make_model_var_climatologies(case_name, cfg, VARS_TO_RUN, sample_years_by_name, years=years))
    if RUN_EPFLUX:
        outputs.append(make_epflux_climatology(case_name, cfg, sample_years_by_name))
    return outputs


validate_inputs()

all_outputs = {}
for case_name in progress_iter(CASES_TO_RUN, desc="Cases", unit="case"):
    all_outputs[case_name] = process_case(case_name)

print("\nDone. Outputs:")
for case_name, outputs in all_outputs.items():
    print(case_name)
    for out in outputs:
        print("  ", out)


DEBUG_MODE = False
OUTPUT_SUBDIR = climatology
CASES_TO_RUN = ['B2000WCN007009010011_Clim3D', 'B2000WCN_NOCOUPL001002', 'B2000WCN001002', 'BWCN']
VARS_TO_RUN = ['U', 'T', 'O3']
RUN_EPFLUX = True
SHOW_PROGRESS = True
MODEL_VAR_SHARED_PRESSURE = True
INTERP_OUTPUT_DTYPE = <class 'numpy.float32'>
DASK_NUM_WORKERS = 8
DASK_CHUNK_SIZE = 96 MiB
CHUNKS_MODEL = auto
CHUNKS_EP = {'time': 365, 'plev': 23, 'lat': 96}
O3_EXTREME_SAMPLE_NAMES = ['low25', 'high25']
O3_EXTREME_RANKING_ROLLING_DAYS = 5
PRESERVE_EXISTING_ALL_CLIMATOLOGY = True
OVERWRITE_EXTREME_CLIMATOLOGIES = False
O3_EXTREME_SAMPLE_NAMES = ['low25_raw', 'low25_rm5', 'high25_raw', 'high25_rm5']
O3_EXTREME_RANKING_METHODS = ['raw', 'rm5']
PRESERVE_EXISTING_ALL_CLIMATOLOGY = True
OVERWRITE_EXTREME_CLIMATOLOGIES = False
DROP_LEGACY_AMBIGUOUS_EXTREMES = True


Checking inputs:   0%|          | 0/4 [00:00<?, ?case/s]

[B2000WCN007009010011_Clim3D] low25_raw years (30_70hPa): n=52, first=[2, 7, 20, 21, 26]
[B2000WCN007009010011_Clim3D] low25_rm5 years (30_70hPa): n=52, first=[2, 20, 21, 26, 30]
[B2000WCN007009010011_Clim3D] high25_raw years (30_70hPa): n=52, first=[9, 15, 17, 23, 25]
[B2000WCN007009010011_Clim3D] high25_rm5 years (30_70hPa): n=52, first=[9, 15, 17, 23, 25]
[B2000WCN007009010011_Clim3D] split EPflux files found: all_waves, wave1, wave2, wave_rest
[B2000WCN_NOCOUPL001002] low25_raw years (30_70hPa): n=51, first=[2, 5, 8, 12, 18]
[B2000WCN_NOCOUPL001002] low25_rm5 years (30_70hPa): n=51, first=[2, 5, 8, 12, 18]
[B2000WCN_NOCOUPL001002] high25_raw years (30_70hPa): n=51, first=[3, 4, 7, 15, 16]
[B2000WCN_NOCOUPL001002] high25_rm5 years (30_70hPa): n=51, first=[3, 4, 7, 15, 16]
[B2000WCN_NOCOUPL001002] split EPflux files found: all_waves, wave1, wave2, wave_rest
[B2000WCN001002] low25_raw years (30_70hPa): n=51, first=[2, 3, 7, 8, 9]
[B2000WCN001002] low25_rm5 years (30_70hPa): n=51, firs

Cases:   0%|          | 0/4 [00:00<?, ?case/s]


CASE: B2000WCN007009010011_Clim3D
ROOT: /mnt/soclim0/public_data/weiji/B2000WCN007009010011_Clim3D_timefixed
[B2000WCN007009010011_Clim3D] U: preserving existing all climatology; updating raw/rm5 O3-extreme composites only
[B2000WCN007009010011_Clim3D] T: preserving existing all climatology; updating raw/rm5 O3-extreme composites only
[B2000WCN007009010011_Clim3D] O3: preserving existing all climatology; updating raw/rm5 O3-extreme composites only


...

  opening EPFLUX wave2: /mnt/soclim0/public_data/weiji/BWCN/EPflux_daily/wave2/EPFLUX_wave2_24yr_time_plev_lat.nc


wave2 EPFLUX vars:   0%|          | 0/4 [00:00<?, ?var/s]

  opening EPFLUX wave_rest: /mnt/soclim0/public_data/weiji/BWCN/EPflux_daily/wave_rest/EPFLUX_wave_rest_24yr_time_plev_lat.nc


wave_rest EPFLUX vars:   0%|          | 0/4 [00:00<?, ?var/s]

[WRITE] /mnt/soclim0/public_data/weiji/BWCN/climatology/EPFLUX_climatology_plev_doy.nc
[COMPUTE] updating raw/rm5 extremes in EPFLUX_climatology_plev_doy.nc
[########################################] | 100% Completed | 50.77 s

Done. Outputs:
B2000WCN007009010011_Clim3D
   /mnt/soclim0/public_data/weiji/B2000WCN007009010011_Clim3D_timefixed/climatology/U_climatology_plev_doy.nc
   /mnt/soclim0/public_data/weiji/B2000WCN007009010011_Clim3D_timefixed/climatology/T_climatology_plev_doy.nc
   /mnt/soclim0/public_data/weiji/B2000WCN007009010011_Clim3D_timefixed/climatology/O3_climatology_plev_doy.nc
   /mnt/soclim0/public_data/weiji/B2000WCN007009010011_Clim3D_timefixed/climatology/EPFLUX_climatology_plev_doy.nc
B2000WCN_NOCOUPL001002
   /mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002_timefixed/climatology/U_climatology_plev_doy.nc
   /mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002_timefixed/climatology/T_climatology_plev_doy.nc
   /mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL